# XR EgoPose Inference Notebook

This notebook demonstrates how to run inference with trained checkpoint and visualize results.

In [ ]:
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import h5py

# Add mmpose to path
sys.path.insert(0, str(Path('.').parent.resolve()))

from mmengine.config import Config
from mmengine.dataset import Compose, pseudo_collate
from mmengine.registry import init_default_scope
from mmengine.runner import load_checkpoint

from mmpose.registry import MODELS

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]

## 1. Configuration

In [ ]:
# Paths - Update these according to your setup
CONFIG_PATH = 'custom_config/HMD_xregopose_h5cache_config.py'
CHECKPOINT_PATH = '../work_dirs/HMD_xregopose_h5cache_test/best_xregopose_Full Body_All_mpjpe_epoch_7.pth'
DATASET_ROOT = r'F:\ego_cam_dataset\Test'

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 2. Load Model

In [ ]:
# Load config
config = Config.fromfile(CONFIG_PATH)
config.model.train_cfg = None

# Initialize mmpose scope
init_default_scope('mmpose')

# Build model
model = MODELS.build(config.model)
model.to(DEVICE)
model.eval()

# Load checkpoint
checkpoint = load_checkpoint(model, CHECKPOINT_PATH, map_location='cpu')
print(f'Loaded checkpoint from {CHECKPOINT_PATH}')

# Get dataset meta info
from mmpose.datasets.datasets.utils import parse_pose_metainfo
metainfo = dict(from_file='../mmpose/datasets/datasets/body3d/egopose_info.py')
dataset_meta = parse_pose_metainfo(metainfo)
model.dataset_meta = dataset_meta

## 3. Define Inference Pipeline

In [ ]:
# Build inference pipeline (includes GenerateTarget for HMD info)
pipeline_cfg = [
    dict(type='LoadImage'),
    dict(padding=1.0, type='GetBBoxCenterScale'),
    dict(input_size=(256, 256), type='TopdownAffine'),
    dict(
        encoder=dict(
            heatmap_size=(47, 47),
            input_size=(256, 256),
            sigma=3,
            type='Custom_mo2cap2_MSRAHeatmap'
        ),
        type='GenerateTarget'
    ),
    dict(type='PackPoseInputs'),
]
pipeline = Compose(pipeline_cfg)

# Skeleton connections for visualization
SKELETON = [
    [0, 1],   # Spine2 -> Head
    [0, 2],   # Spine2 -> LeftArm
    [2, 3],   # LeftArm -> LeftForeArm
    [3, 4],   # LeftForeArm -> LeftHand
    [0, 5],   # Spine2 -> RightArm
    [5, 6],   # RightArm -> RightForeArm
    [6, 7],   # RightForeArm -> RightHand
    [0, 8],   # Spine2 -> LeftUpLeg
    [8, 9],   # LeftUpLeg -> LeftLeg
    [9, 10],  # LeftLeg -> LeftFoot
    [10, 11], # LeftFoot -> LeftToeBase
    [0, 12],  # Spine2 -> RightUpLeg
    [12, 13], # RightUpLeg -> RightLeg
    [13, 14], # RightLeg -> RightFoot
    [14, 15], # RightFoot -> RightToeBase
]

JOINT_NAMES = [
    'Spine2', 'Head', 'LeftArm', 'LeftForeArm', 'LeftHand',
    'RightArm', 'RightForeArm', 'RightHand',
    'LeftUpLeg', 'LeftLeg', 'LeftFoot', 'LeftToeBase',
    'RightUpLeg', 'RightLeg', 'RightFoot', 'RightToeBase'
]

# Colors
KPT_COLORS = np.array([
    [51, 153, 255], [51, 153, 255], [51, 153, 255], [0, 255, 0], [0, 255, 0],
    [51, 153, 255], [255, 128, 0], [255, 128, 0],
    [51, 153, 255], [0, 255, 0], [0, 255, 0], [0, 255, 0],
    [51, 153, 255], [255, 128, 0], [255, 128, 0], [255, 128, 0]
]) / 255.0

LINK_COLORS = np.array([
    [51, 153, 255], [51, 153, 255], [0, 255, 0], [0, 255, 0],
    [51, 153, 255], [255, 128, 0], [255, 128, 0],
    [51, 153, 255], [0, 255, 0], [0, 255, 0], [0, 255, 0],
    [51, 153, 255], [255, 128, 0], [255, 128, 0], [255, 128, 0]
]) / 255.0

## 4. Load Test Data from H5 Cache

In [ ]:
# Load cache
cache_file = os.path.join(DATASET_ROOT, 'annotations_cache.h5')
print(f'Loading cache from: {cache_file}')

with h5py.File(cache_file, 'r') as hf:
    img_paths = hf['img_paths'][:]
    hmd_infos = hf['hmd_info'][:]
    keypoints_3d_gt = hf['keypoint3d'][:]
    keypoints_2d = hf['keypoints'][:]
    actions = hf['actions'][:]

print(f'Total samples: {len(img_paths)}')
print(f'HMD info shape: {hmd_infos.shape}')
print(f'3D keypoints shape: {keypoints_3d_gt.shape}')
print(f'2D keypoints shape: {keypoints_2d.shape}')

## 5. Inference Function

In [ ]:
def run_inference(img_path, hmd_info, kpts_2d, kpts_3d):
    """Run inference on a single image.
    
    Args:
        img_path: Path to input image
        hmd_info: HMD info array with shape (1, 9)
        kpts_2d: 2D keypoints with shape (1, 16, 2)
        kpts_3d: 3D keypoints with shape (1, 16, 3) - needed for encoder
        
    Returns:
        Predicted 3D keypoints with shape (16, 3)
    """
    # Decode path if bytes
    if isinstance(img_path, bytes):
        img_path = img_path.decode('utf-8')
    
    # Load image
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f'Could not load image: {img_path}')
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Prepare data
    data_info = {
        'img_path': img_path,
        'img': img_rgb,
        'bbox': np.array([[0, 0, w, h]], dtype=np.float32),
        'bbox_score': np.array([1.0], dtype=np.float32),
        'keypoints': kpts_2d,
        'keypoints_visible': np.ones((1, 16), dtype=np.float32),
        'keypoint3d': kpts_3d,
    }
    data_info.update(dataset_meta)
    
    # Apply pipeline
    data = pipeline(data_info)
    
    # Add HMD info
    data['data_samples'].gt_instance_labels.set_field(
        torch.from_numpy(hmd_info.astype(np.float32)), 'hmd_info'
    )
    
    # Create batch
    batch = pseudo_collate([data])
    
    # Run inference
    with torch.no_grad():
        results = model.test_step(batch)
    
    pred_instances = results[0].pred_instances
    pred_3d = pred_instances.keypoint_3d.cpu().numpy()[0]  # (16, 3)
    
    return img_rgb, pred_3d

## 6. Visualization Functions

In [ ]:
def draw_3d_skeleton(ax, keypoints, title='3D Pose', color_kpts=True):
    """Draw 3D skeleton on matplotlib axis."""
    # Draw keypoints
    if color_kpts:
        ax.scatter(keypoints[:, 0], keypoints[:, 1], keypoints[:, 2],
                   c=KPT_COLORS, s=50, marker='o')
    else:
        ax.scatter(keypoints[:, 0], keypoints[:, 1], keypoints[:, 2],
                   c='blue', s=50, marker='o')
    
    # Draw skeleton
    for idx, (i, j) in enumerate(SKELETON):
        pts = keypoints[[i, j]]
        ax.plot(pts[:, 0], pts[:, 1], pts[:, 2],
                color=LINK_COLORS[idx] if color_kpts else 'blue', linewidth=2)
    
    # Set axis properties
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    
    # Set equal aspect ratio
    center = keypoints.mean(axis=0)
    max_range = np.abs(keypoints - center).max() * 1.2
    ax.set_xlim([center[0] - max_range, center[0] + max_range])
    ax.set_ylim([center[1] - max_range, center[1] + max_range])
    ax.set_zlim([center[2] - max_range, center[2] + max_range])
    ax.view_init(elev=15, azim=70)


def visualize_result(img, pred_3d, gt_3d=None):
    """Visualize inference result with image and 3D pose."""
    num_cols = 3 if gt_3d is not None else 2
    fig = plt.figure(figsize=(6 * num_cols, 6))
    
    # Original image
    ax1 = fig.add_subplot(1, num_cols, 1)
    ax1.imshow(img)
    ax1.set_title('Input Image')
    ax1.axis('off')
    
    # Predicted 3D pose
    ax2 = fig.add_subplot(1, num_cols, 2, projection='3d')
    draw_3d_skeleton(ax2, pred_3d, 'Predicted 3D Pose')
    
    # Ground truth 3D pose
    if gt_3d is not None:
        ax3 = fig.add_subplot(1, num_cols, 3, projection='3d')
        draw_3d_skeleton(ax3, gt_3d, 'Ground Truth 3D Pose')
    
    plt.tight_layout()
    plt.show()


def compute_mpjpe(pred, gt):
    """Compute Mean Per Joint Position Error (MPJPE)."""
    return np.mean(np.sqrt(np.sum((pred - gt) ** 2, axis=-1)))

## 7. Run Inference on Sample

In [ ]:
# Select random sample
sample_idx = np.random.randint(0, len(img_paths))

img_path = img_paths[sample_idx]
hmd_info = hmd_infos[sample_idx]
kpts_2d = keypoints_2d[sample_idx]
gt_3d = keypoints_3d_gt[sample_idx][0]  # (16, 3)
action = actions[sample_idx].decode('utf-8') if isinstance(actions[sample_idx], bytes) else actions[sample_idx]

print(f'Sample {sample_idx}')
print(f'Image: {img_path}')
print(f'Action: {action}')

# Run inference
img_rgb, pred_3d = run_inference(img_path, hmd_info, kpts_2d, keypoints_3d_gt[sample_idx])

# Compute MPJPE
mpjpe = compute_mpjpe(pred_3d, gt_3d)
print(f'MPJPE: {mpjpe:.4f} m')

# Visualize
visualize_result(img_rgb, pred_3d, gt_3d)

## 8. Run Inference on Multiple Samples

In [ ]:
# Run inference on multiple random samples
num_samples = 5
sample_indices = np.random.choice(len(img_paths), num_samples, replace=False)

mpjpe_list = []

for idx in sample_indices:
    img_path = img_paths[idx]
    hmd_info = hmd_infos[idx]
    kpts_2d = keypoints_2d[idx]
    gt_3d = keypoints_3d_gt[idx][0]
    action = actions[idx].decode('utf-8') if isinstance(actions[idx], bytes) else actions[idx]
    
    try:
        img_rgb, pred_3d = run_inference(img_path, hmd_info, kpts_2d, keypoints_3d_gt[idx])
        mpjpe = compute_mpjpe(pred_3d, gt_3d)
        mpjpe_list.append(mpjpe)
        
        print(f'\n--- Sample {idx} ---')
        print(f'Action: {action}')
        print(f'MPJPE: {mpjpe:.4f} m')
        
        visualize_result(img_rgb, pred_3d, gt_3d)
    except Exception as e:
        print(f'Error processing sample {idx}: {e}')

print(f'\n=== Summary ===')
print(f'Average MPJPE: {np.mean(mpjpe_list):.4f} m')
print(f'Std MPJPE: {np.std(mpjpe_list):.4f} m')

## 9. Interactive 3D Visualization

In [ ]:
# Interactive 3D visualization with rotation
from ipywidgets import interact, IntSlider, FloatSlider

# Get a sample
sample_idx = np.random.randint(0, len(img_paths))
img_rgb, pred_3d = run_inference(img_paths[sample_idx], hmd_infos[sample_idx], 
                                  keypoints_2d[sample_idx], keypoints_3d_gt[sample_idx])
gt_3d = keypoints_3d_gt[sample_idx][0]

def plot_3d_interactive(elev=15, azim=70):
    fig = plt.figure(figsize=(12, 5))
    
    ax1 = fig.add_subplot(121, projection='3d')
    draw_3d_skeleton(ax1, pred_3d, 'Predicted')
    ax1.view_init(elev=elev, azim=azim)
    
    ax2 = fig.add_subplot(122, projection='3d')
    draw_3d_skeleton(ax2, gt_3d, 'Ground Truth')
    ax2.view_init(elev=elev, azim=azim)
    
    plt.tight_layout()
    plt.show()

interact(plot_3d_interactive, 
         elev=IntSlider(min=-90, max=90, step=5, value=15),
         azim=IntSlider(min=0, max=360, step=10, value=70))

## 10. Per-Joint Error Analysis

In [ ]:
# Analyze per-joint errors
num_samples_analysis = 100
sample_indices = np.random.choice(len(img_paths), num_samples_analysis, replace=False)

joint_errors = []

print('Running inference on', num_samples_analysis, 'samples...')
for i, idx in enumerate(sample_indices):
    if i % 20 == 0:
        print(f'  Processing {i}/{num_samples_analysis}')
    try:
        img_rgb, pred_3d = run_inference(img_paths[idx], hmd_infos[idx],
                                          keypoints_2d[idx], keypoints_3d_gt[idx])
        gt_3d = keypoints_3d_gt[idx][0]
        
        # Per-joint error
        per_joint = np.sqrt(np.sum((pred_3d - gt_3d) ** 2, axis=-1))
        joint_errors.append(per_joint)
    except:
        pass

joint_errors = np.array(joint_errors)  # (N, 16)
mean_per_joint = joint_errors.mean(axis=0)
std_per_joint = joint_errors.std(axis=0)

# Plot per-joint error
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(JOINT_NAMES))
bars = ax.bar(x, mean_per_joint * 100, yerr=std_per_joint * 100, capsize=3)

# Color bars by body part
colors = ['blue', 'blue', 'green', 'green', 'green', 'orange', 'orange', 'orange',
          'purple', 'purple', 'purple', 'purple', 'red', 'red', 'red', 'red']
for bar, color in zip(bars, colors):
    bar.set_color(color)

ax.set_xticks(x)
ax.set_xticklabels(JOINT_NAMES, rotation=45, ha='right')
ax.set_ylabel('Error (cm)')
ax.set_title(f'Per-Joint Position Error (n={len(joint_errors)} samples)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('\nPer-joint error (cm):')
for name, err, std in zip(JOINT_NAMES, mean_per_joint * 100, std_per_joint * 100):
    print(f'  {name:15s}: {err:.2f} +/- {std:.2f}')

## 11. Save Results

In [ ]:
# Save visualization results to disk
output_dir = 'inference_results'
os.makedirs(output_dir, exist_ok=True)

num_to_save = 10
sample_indices = np.random.choice(len(img_paths), num_to_save, replace=False)

for i, idx in enumerate(sample_indices):
    try:
        img_rgb, pred_3d = run_inference(img_paths[idx], hmd_infos[idx],
                                          keypoints_2d[idx], keypoints_3d_gt[idx])
        gt_3d = keypoints_3d_gt[idx][0]
        
        # Create figure
        fig = plt.figure(figsize=(18, 6))
        
        ax1 = fig.add_subplot(131)
        ax1.imshow(img_rgb)
        ax1.set_title('Input Image')
        ax1.axis('off')
        
        ax2 = fig.add_subplot(132, projection='3d')
        draw_3d_skeleton(ax2, pred_3d, 'Predicted')
        
        ax3 = fig.add_subplot(133, projection='3d')
        draw_3d_skeleton(ax3, gt_3d, 'Ground Truth')
        
        mpjpe = compute_mpjpe(pred_3d, gt_3d)
        fig.suptitle(f'MPJPE: {mpjpe:.4f} m', fontsize=14)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'result_{i:04d}.png'), dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f'Saved result {i+1}/{num_to_save}')
    except Exception as e:
        print(f'Error saving sample {idx}: {e}')

print(f'\nResults saved to {output_dir}/')